# Bronze Layer: Ingest DataBursátil API
## 📥 Raw Data Ingestion Pipeline

**Purpose:** Fetch raw market data from DataBursátil API and store in `bronze.api_responses`

**Key Features:**
- Ticker-agnostic design (iterates over `silver.tickers`)
- Stores raw JSON responses for full audit trail
- Idempotent loading with metadata tracking
- Error handling and retry logic
- Versioned and reproducible

**Outputs:**
- `bronze.api_responses` - Raw API responses with metadata

---

### Architecture: Medallion Bronze Layer
```
DataBursátil API → bronze.api_responses (Delta Lake)
                    ↓
              [Raw JSON stored]
```

## 1️⃣ Setup and Initialization

In [ ]:
# Install required packages (if not in cluster libraries)
%pip install requests pyyaml pandas --quiet

# Restart Python to use new packages
dbutils.library.restartPython()

In [ ]:
import requests
import json
import uuid
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Any
import time

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

print("✓ Libraries imported successfully")
print(f"✓ Spark version: {spark.version}")
print(f"✓ Notebook started at: {datetime.now(timezone.utc).isoformat()}")

In [ ]:
# Get API token from Databricks Secrets
# IMPORTANT: First create the secret scope in Databricks:
# databricks secrets create-scope --scope databursatil
# databricks secrets put --scope databursatil --key api_token

try:
    API_TOKEN = dbutils.secrets.get(
        scope=CONFIG["databursatil"]["secret_scope"], 
        key=CONFIG["databursatil"]["secret_key"]
    )
    print("✓ API token retrieved from secrets")
except Exception as e:
    print(f"⚠️  Failed to get secret: {e}")
    print("   Using placeholder token for testing")
    API_TOKEN = "YOUR_DATABURSATIL_TOKEN_HERE"  # Replace for local testing

## 2️⃣ DataBursátil API Client

Based on DataBursátil API documentation:
- Endpoint: `https://api.databursatil.com/api/v1/historical`
- Authentication: Bearer token in headers
- Parameters: `ticker`, `start_date`, `end_date`, `interval`
- Response: JSON with OHLCV data

In [ ]:
class DataBursatilClient:
    """Client for DataBursátil API with retry logic and error handling."""
    
    def __init__(self, api_token: str, base_url: str):
        self.api_token = api_token
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
            "User-Agent": "Databricks-ETL/1.0"
        })
    
    def get_historical_data(
        self, 
        ticker: str, 
        start_date: str, 
        end_date: str,
        interval: str = "1D"
    ) -> Dict[str, Any]:
        """
        Fetch historical OHLCV data for a ticker.
        
        Args:
            ticker: Ticker symbol (e.g., "AMXL", "WALMEX")
            start_date: Start date in YYYY-MM-DD format
            end_date: End date in YYYY-MM-DD format
            interval: Data interval (1D, 1H, etc.)
            
        Returns:
            Dictionary with 'success', 'data', 'status_code', 'headers', 'error'
        """
        endpoint = f"{self.base_url}/api/v1/historical"
        
        params = {
            "ticker": ticker,
            "start_date": start_date,
            "end_date": end_date,
            "interval": interval,
            "country": "MX",
            "market": "BMV"
        }
        
        for attempt in range(CONFIG["ingestion"]["retry_attempts"]):
            try:
                response = self.session.get(endpoint, params=params, timeout=30)
                
                result = {
                    "success": response.status_code == 200,
                    "status_code": response.status_code,
                    "headers": dict(response.headers),
                    "data": response.text,
                    "error": None
                }
                
                if response.status_code == 200:
                    try:
                        # Validate JSON
                        json.loads(response.text)
                        return result
                    except json.JSONDecodeError as e:
                        result["success"] = False
                        result["error"] = f"Invalid JSON: {str(e)}"
                        return result
                elif response.status_code == 429:  # Rate limit
                    wait_time = int(response.headers.get("Retry-After", 60))
                    print(f"   Rate limited. Waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    result["error"] = f"HTTP {response.status_code}: {response.text[:200]}"
                    return result
                    
            except requests.exceptions.Timeout:
                if attempt < CONFIG["ingestion"]["retry_attempts"] - 1:
                    print(f"   Timeout on attempt {attempt + 1}, retrying...")
                    time.sleep(CONFIG["ingestion"]["retry_delay_seconds"])
                    continue
                else:
                    return {
                        "success": False,
                        "status_code": None,
                        "headers": {},
                        "data": None,
                        "error": "Request timeout after retries"
                    }
            except Exception as e:
                return {
                    "success": False,
                    "status_code": None,
                    "headers": {},
                    "data": None,
                    "error": f"Exception: {str(e)}"
                }
        
        return {
            "success": False,
            "status_code": None,
            "headers": {},
            "data": None,
            "error": "Max retries exceeded"
        }

# Initialize API client
api_client = DataBursatilClient(API_TOKEN, CONFIG["databursatil"]["base_url"])
print("✓ DataBursátil API client initialized")

In [ ]:
# Check if silver.tickers table exists
try:
    active_tickers_df = spark.sql("""
        SELECT ticker, ticker_name, market
        FROM silver.tickers
        WHERE is_active = true
        ORDER BY ticker
    """)
    
    active_tickers = [row.ticker for row in active_tickers_df.collect()]
    
    print(f"✓ Found {len(active_tickers)} active tickers:")
    for ticker in active_tickers:
        print(f"   - {ticker}")
    
except Exception as e:
    print(f"⚠️  silver.tickers table not found or empty: {e}")
    print("   Using default tickers for initial load")
    
    # Default tickers for Mexican market (BMV)
    active_tickers = [
        "AMXL",    # América Móvil
        "WALMEX",  # Walmart de México
        "GFNORTEO", # Grupo Financiero Banorte
        "CEMEXCPO", # CEMEX
        "ALFAA",   # Alfa
        "FEMSAUBD", # FEMSA
        "GMEXICOB", # Grupo México
        "BIMBOA",  # Bimbo
        "AC"       # Arca Continental
    ]
    
    print(f"✓ Using {len(active_tickers)} default tickers")
    
print(f"\n📊 Tickers to process: {', '.join(active_tickers)}")

In [ ]:
# Calculate date range
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=CONFIG["ingestion"]["lookback_days"])

print(f"📅 Fetching data from {start_date} to {end_date}")
print(f"   ({CONFIG['ingestion']['lookback_days']} days lookback)\n")

# Prepare list to collect all API responses
api_responses = []

# Fetch data for each ticker
for i, ticker in enumerate(active_tickers, 1):
    print(f"[{i}/{len(active_tickers)}] Fetching {ticker}...", end=" ")
    
    # Call API
    response = api_client.get_historical_data(
        ticker=ticker,
        start_date=start_date.isoformat(),
        end_date=end_date.isoformat(),
        interval="1D"
    )
    
    # Create record for bronze table
    record = {
        "response_id": str(uuid.uuid4()),
        "endpoint": "/api/v1/historical",
        "request_params": {
            "ticker": ticker,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "interval": "1D",
            "market": "BMV"
        },
        "response_body": response["data"],
        "response_status": response["status_code"],
        "response_headers": response["headers"],
        "ingestion_timestamp": datetime.now(timezone.utc),
        "data_date": end_date,  # Partition by end date
        "ticker": ticker,
        "is_successful": response["success"],
        "error_message": response["error"]
    }
    
    api_responses.append(record)
    
    if response["success"]:
        print("✓")
    else:
        print(f"❌ {response['error']}")
    
    # Rate limiting - be nice to the API
    time.sleep(0.5)

print(f"\n✓ Fetched {len(api_responses)} API responses")
print(f"  Successful: {sum(1 for r in api_responses if r['is_successful'])}")
print(f"  Failed: {sum(1 for r in api_responses if not r['is_successful'])}")

In [ ]:
# Write to bronze.api_responses (append mode for idempotency)
print("💾 Writing to bronze.api_responses...")

responses_df.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("data_date") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.api_responses")

print("✓ Data written to bronze.api_responses")

# Show summary
spark.sql("""
    SELECT 
        data_date,
        COUNT(*) as total_responses,
        SUM(CASE WHEN is_successful THEN 1 ELSE 0 END) as successful,
        SUM(CASE WHEN NOT is_successful THEN 1 ELSE 0 END) as failed
    FROM bronze.api_responses
    WHERE data_date >= current_date() - INTERVAL 7 DAYS
    GROUP BY data_date
    ORDER BY data_date DESC
""").show()

In [ ]:
# Sample data check
print("📊 Sample data from bronze.api_responses:\n")

sample_df = spark.sql("""
    SELECT 
        response_id,
        ticker,
        endpoint,
        data_date,
        is_successful,
        response_status,
        ingestion_timestamp,
        SUBSTRING(response_body, 1, 100) as response_preview
    FROM bronze.api_responses
    ORDER BY ingestion_timestamp DESC
    LIMIT 5
""")

sample_df.show(truncate=False)

print("\n✅ Bronze layer ingestion complete!")

## 5️⃣ Summary and Next Steps

**Completion Summary:**
- ✅ Fetched data from DataBursátil API
- ✅ Stored raw JSON in bronze.api_responses
- ✅ Partitioned by data_date for efficient querying
- ✅ Tracked success/failure for each ticker

**Next Steps:**
1. Run notebook `02_normalize_to_silver.ipynb` to parse JSON and normalize data
2. Check for failed responses and retry if needed
3. Set up Databricks Workflow to run this notebook daily

**Reproducibility:**
- All raw responses are stored with metadata
- Can reprocess any date by querying bronze layer
- Versioned in Delta Lake with full history

In [ ]:
# Define schema for bronze.api_responses
bronze_schema = StructType([
    StructField("response_id", StringType(), False),
    StructField("endpoint", StringType(), False),
    StructField("request_params", MapType(StringType(), StringType()), True),
    StructField("response_body", StringType(), True),
    StructField("response_status", IntegerType(), True),
    StructField("response_headers", MapType(StringType(), StringType()), True),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("data_date", DateType(), True),
    StructField("ticker", StringType(), True),
    StructField("is_successful", BooleanType(), True),
    StructField("error_message", StringType(), True)
])

# Create DataFrame
responses_df = spark.createDataFrame(api_responses, schema=bronze_schema)

print(f"✓ Created DataFrame with {responses_df.count()} rows")
responses_df.printSchema()

## 4️⃣ Ingest Data to Bronze Layer

Fetch data from API and store raw responses in `bronze.api_responses` table.

## 3️⃣ Get Active Tickers

Read from `silver.tickers` to get list of active tickers to ingest.
This implements the ticker-agnostic design - add new tickers via SQL INSERT.

In [ ]:
# Configuration
CONFIG = {
    "databursatil": {
        "base_url": "https://api.databursatil.com",
        "secret_scope": "databursatil",  # Create this in Databricks Secrets
        "secret_key": "api_token",
        "endpoints": {
            "historical": "/api/v1/historical",
            "tickers": "/api/v1/tickers",
            "quote": "/api/v1/quote"
        },
        "default_params": {
            "country": "MX",
            "market": "BMV",
            "interval": "1D"
        }
    },
    "bronze": {
        "table": "bronze.api_responses"
    },
    "silver": {
        "tickers_table": "silver.tickers"
    },
    "ingestion": {
        "lookback_days": 30,  # How many days of historical data to fetch
        "batch_size": 10,     # Process tickers in batches
        "retry_attempts": 3,
        "retry_delay_seconds": 2
    }
}

print("✓ Configuration loaded")